# RDD Transformation and Actions
This lab demonstrates RDD basic transformation and action operations

In [45]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local').getOrCreate()
sc = spark.sparkContext

## Simple RDD parallelize, sum of 1+2+3+4...+100

In [46]:
rdd = sc.parallelize(range(100 + 1))
rdd.sum()

5050

## RDD can also parallelize array of words, and count words

In [47]:
words = sc.parallelize (
   ["scala", 
   "java", 
   "hadoop", 
   "spark", 
   "akka",
   "spark vs hadoop", 
   "pyspark",
   "pyspark and spark",
   "java",
   "spark",
   "scala",
   "hadoop",
   "pyspark"]
)
counts = words.count()  # count() is an action
print("Number of elements in RDD -> %i" % (counts))

Number of elements in RDD -> 13


## You can use collect operation to get elements from the RDD

In [48]:
coll = words.collect() # collect() is another action
print("Elements in RDD -> %s" % (coll))

Elements in RDD -> ['scala', 'java', 'hadoop', 'spark', 'akka', 'spark vs hadoop', 'pyspark', 'pyspark and spark', 'java', 'spark', 'scala', 'hadoop', 'pyspark']


# Transformation

## 1. map

In [49]:
words_map = words.map(lambda x: (x, 1))

In [50]:
mapping = words_map.collect()
print("Key value pair -> %s" % (mapping))

Key value pair -> [('scala', 1), ('java', 1), ('hadoop', 1), ('spark', 1), ('akka', 1), ('spark vs hadoop', 1), ('pyspark', 1), ('pyspark and spark', 1), ('java', 1), ('spark', 1), ('scala', 1), ('hadoop', 1), ('pyspark', 1)]


## The "Reduce" operation of RDD
reduceByKey

In [51]:
reducing = words_map.reduceByKey(lambda a, b: a + b)
reduce_output = reducing.collect()
print(reduce_output)

[('scala', 2), ('java', 2), ('hadoop', 2), ('spark', 2), ('akka', 1), ('spark vs hadoop', 1), ('pyspark', 2), ('pyspark and spark', 1)]


## 2 Filter

In [8]:
rdd1 = sc.parallelize([1,2,3,4,5,6])
rdd2 = rdd1.filter(lambda x: x % 2 == 0)  # only even numbers are selected
print("After filtering odd number ->%s"  % rdd2.collect())

After filtering odd number ->[2, 4, 6]


## 3 flatMap 

In [9]:
rdd1 = sc.parallelize(['hello', 'You    are very good'])
new_rdd1 = rdd1.flatMap(lambda x: x.split())
print('After flatMAp -> ', new_rdd1.collect())

After flatMAp ->  ['hello', 'You', 'are', 'very', 'good']


## 4 Sample 

In [55]:
py_number_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
number_rdd = spark.sparkContext.parallelize(py_number_list)

sample_number_rdd = number_rdd.sample(withReplacement=False, fraction=0.4)
print(sample_number_rdd.collect())

[3, 4, 6, 9]


## 5 Union 

In [56]:
x = sc.parallelize(['A','A','B'])
y = sc.parallelize(['D','C','A'])
z = x.union(y)
print(z.collect())
#print(x.collect())

['A', 'A', 'B', 'D', 'C', 'A']


In [57]:
x1 = sc.parallelize(['A','A','B'])
y1 = sc.parallelize([['1', '2'], ['3', '4'], ['5', '6']])
z1 = x1.union(y1)
print(z1.collect())

['A', 'A', 'B', ['1', '2'], ['3', '4'], ['5', '6']]


## 6 Distinct

In [58]:
py_number_list = [1, 3, 2, 1, 2, 6, 1, 6, 3, 10]
number_rdd = sc.parallelize(py_number_list)
distinct_number_rdd = number_rdd.distinct()
print(distinct_number_rdd.collect())

[1, 3, 2, 6, 10]


## 7 groupByKey

In [16]:
def fun(x):
   k,v=x
   d=dict()
   for i in v:
      d.update(i)
   return k,d

rdd1 = sc.parallelize([("a", {"a":12}), ("b", {"a1":45}), ("a", {"a2":45})])
rdd2 = rdd1.groupByKey()
print(rdd2.collect())

rdd3 = rdd2.map(lambda x:fun(x))
print(rdd3.collect())

[('a', <pyspark.resultiterable.ResultIterable object at 0x000001D23F113D30>), ('b', <pyspark.resultiterable.ResultIterable object at 0x000001D23F1302E0>)]
[('a', {'a': 12, 'a2': 45}), ('b', {'a1': 45})]


## 8 reduceByKey

In [59]:
rdd1 = sc.parallelize([('a',1), ('b', 100), ('a', 300), ('b', 3), ('a', 200)])
rdd2 = rdd1.reduceByKey(lambda v1, v2: v1+v2)
print(rdd2.collect())

[('a', 501), ('b', 103)]


In [60]:
#user defined function
def add(a,b):
   return a+b

rdd3 = rdd1.reduceByKey(add)
print(rdd3.collect())

[('a', 501), ('b', 103)]


## 9 sortByKey

In [63]:
rdd1 = sc.parallelize([(4, 2582), (3, 3222), (1, 4190), (5, 2502), (2, 2537)])
print(rdd1.collect())

rdd2 = rdd1.sortByKey(ascending=True)
print(rdd2.collect())

rdd3 = rdd1.sortByKey(ascending=False)
print(rdd3.collect())


[(4, 2582), (3, 3222), (1, 4190), (5, 2502), (2, 2537)]
[(1, 4190), (2, 2537), (3, 3222), (4, 2582), (5, 2502)]
[(5, 2502), (4, 2582), (3, 3222), (2, 2537), (1, 4190)]


## 10 Join 
joining elements containing matching keys

In [64]:
x = sc.parallelize([("spark", 1), ("hadoop", 4)])
y = sc.parallelize([("spark", 2), ("hadoop", 5)])
joined = x.join(y)
final = joined.collect()
print("Join RDD -> %s" % (final))

Join RDD -> [('hadoop', (4, 5)), ('spark', (1, 2))]


In [66]:
body_info = [["Bom", 20, 97.6, 165],
             ["Alice", 23, 90.0, 160],
             ["kuke", 33, 190.0, 170],
             ["jike", 19, 120.0, 170],
             ["Joe", 24, 89.0, 162]]

body_df = spark.createDataFrame(body_info, ["name", "age", "weight", "height"])
body_df.show()

score_info = [["Bom", 88, 97, 90],
             ["Alice", 85, 99, 92],
             ["kuke", 77, 82, 80],
             ["jike", 65, 58, 30],
             ["Joe", 90, 100, 92]]
score_df = spark.createDataFrame(score_info, ["name", "Chinese", "Math", "English"])
score_df.show()

# same as database inner join on statement
merge_df = body_df.join(score_df, on="name", how="inner")
merge_df.show()

+-----+---+------+------+
| name|age|weight|height|
+-----+---+------+------+
|  Bom| 20|  97.6|   165|
|Alice| 23|  90.0|   160|
| kuke| 33| 190.0|   170|
| jike| 19| 120.0|   170|
|  Joe| 24|  89.0|   162|
+-----+---+------+------+

+-----+-------+----+-------+
| name|Chinese|Math|English|
+-----+-------+----+-------+
|  Bom|     88|  97|     90|
|Alice|     85|  99|     92|
| kuke|     77|  82|     80|
| jike|     65|  58|     30|
|  Joe|     90| 100|     92|
+-----+-------+----+-------+

+-----+---+------+------+-------+----+-------+
| name|age|weight|height|Chinese|Math|English|
+-----+---+------+------+-------+----+-------+
|Alice| 23|  90.0|   160|     85|  99|     92|
|  Bom| 20|  97.6|   165|     88|  97|     90|
|  Joe| 24|  89.0|   162|     90| 100|     92|
| jike| 19| 120.0|   170|     65|  58|     30|
| kuke| 33| 190.0|   170|     77|  82|     80|
+-----+---+------+------+-------+----+-------+



In [22]:
rdd = merge_df.rdd
print(rdd.collect())

[Row(name='Alice', age=23, weight=90.0, height=160, Chinese=85, Math=99, English=92), Row(name='Bom', age=20, weight=97.6, height=165, Chinese=88, Math=97, English=90), Row(name='Joe', age=24, weight=89.0, height=162, Chinese=90, Math=100, English=92), Row(name='jike', age=19, weight=120.0, height=170, Chinese=65, Math=58, English=30), Row(name='kuke', age=33, weight=190.0, height=170, Chinese=77, Math=82, English=80)]


In [23]:
d1 = sc.parallelize([(1,2),(2,3),(2,4),(3,4)])
d2 = sc.parallelize([(1,'a'),(2,'b'),(1,'d'),(5,'2')])
d3 = d1.join(d2)
print(d3.collect())

[(2, (3, 'b')), (2, (4, 'b')), (1, (2, 'a')), (1, (2, 'd'))]


## 11 cogroup
def cogroup[W](other: RDD[(K, W)]): RDD[(K, (Iterable[V], Iterable[W]))]

In [24]:
x = sc.parallelize([("a", 1), ("b", 4)])
y = sc.parallelize([("a", 2)])
z = sorted(x.cogroup(y).collect())
print(z)

for i in z:
     for j in i[1]:
         print("key=%s" %i[0],  list(j))


[('a', (<pyspark.resultiterable.ResultIterable object at 0x000001D24038FEB0>, <pyspark.resultiterable.ResultIterable object at 0x000001D24038FF10>)), ('b', (<pyspark.resultiterable.ResultIterable object at 0x000001D23F122AF0>, <pyspark.resultiterable.ResultIterable object at 0x000001D24038FE50>))]
key=a [1]
key=a [2]
key=b [4]
key=b []


## 12 cartesian

In [67]:
rdd1 = sc.parallelize([1,2,4,5,2,3])
rdd2 = sc.parallelize([4,6,5,7,8,6])
rdd3 = rdd1.cartesian(rdd2)
print(rdd3.collect())

[(1, 4), (1, 6), (1, 5), (1, 7), (1, 8), (1, 6), (2, 4), (2, 6), (2, 5), (2, 7), (2, 8), (2, 6), (4, 4), (4, 6), (4, 5), (4, 7), (4, 8), (4, 6), (5, 4), (5, 6), (5, 5), (5, 7), (5, 8), (5, 6), (2, 4), (2, 6), (2, 5), (2, 7), (2, 8), (2, 6), (3, 4), (3, 6), (3, 5), (3, 7), (3, 8), (3, 6)]


# Actions
## 1 reduce

In [69]:
rdd = sc.parallelize([1,2,3,4,5,6])
total = rdd.reduce(lambda x,y:x+y)
print(total)

21


In [70]:
from operator import add

nums = sc.parallelize([1, 2, 3, 4, 5, 6])
adding = nums.reduce(add)
print("Adding all the elements -> %i" % (adding))

Adding all the elements -> 21


## 2 collect

In [71]:
print(rdd.collect())
rdd3 = rdd.map(lambda x: x+2)
print(rdd3.collect())

[1, 2, 3, 4, 5, 6]
[3, 4, 5, 6, 7, 8]


## 3 count and countByValue

In [72]:
print(rdd.count())
print(rdd.countByValue())

6
defaultdict(<class 'int'>, {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1})


## 4 reduceByKey

In [73]:
rdd1 = sc.parallelize((('1',11),('1',2),('11',3),('11',1)))
rdd2 = rdd1.reduceByKey(lambda x,y:x+y)
print(rdd2.collect())

[('1', 13), ('11', 4)]


## 5 first and take(n)

In [74]:
body_info = [["Bom", 20, 97.6, 165],
             ["Alice", 23, 90.0, 160],
             ["kuke", 33, 190.0, 170],
             ["jike", 19, 120.0, 170],
             ["Joe", 24, 89.0, 162]]

body_df = spark.createDataFrame(body_info, ["name", "age", "weight", "height"])
body_df.show()
print(body_df.first())

+-----+---+------+------+
| name|age|weight|height|
+-----+---+------+------+
|  Bom| 20|  97.6|   165|
|Alice| 23|  90.0|   160|
| kuke| 33| 190.0|   170|
| jike| 19| 120.0|   170|
|  Joe| 24|  89.0|   162|
+-----+---+------+------+

Row(name='Bom', age=20, weight=97.6, height=165)


In [75]:
# take(n)
print(body_df.take(2))

[Row(name='Bom', age=20, weight=97.6, height=165), Row(name='Alice', age=23, weight=90.0, height=160)]


## 6 takeSample(withReplacement, fraction, seed)

In [76]:
rdd1 = sc.parallelize(range(0,20))
rdd2 = rdd1.takeSample(True, 6, 1)
print(rdd1.collect())
print(rdd2)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[10, 1, 5, 0, 17, 10]


## 7 countByKey

In [77]:
rdd1 = sc.parallelize((('1',11),('1',2),('11',3),('11',1)))
print(rdd1.countByKey())
print(rdd1.countByKey().items())

defaultdict(<class 'int'>, {'1': 2, '11': 2})
dict_items([('1', 2), ('11', 2)])


## 8 reduceByKey

In [78]:
rdd2 = rdd1.reduceByKey(lambda x,y:x+y)
print(rdd2.collect())
#print(rdd1.reduceByKey(lambda x,y:x+y).collect())

[('1', 13), ('11', 4)]


In [79]:
rdd = sc.parallelize([("foo", 1), ("foo", 2), ("bar", 3)])

print(rdd.reduceByKey(lambda x,y:x+y).collect()) # Sum for each key
# Gives [('foo', 3), ('bar', 3)]

print(rdd.reduceByKey(max).collect()) # Max for each key
# Gives [('foo', 2), ('bar', 3)]

[('foo', 3), ('bar', 3)]
[('foo', 2), ('bar', 3)]


## 9 foreach

In [38]:
def fun(x):
    print(x)

rdd1 = sc.parallelize((('1',11),('1',2),('11',3),('11',1)))
print("Print elements of rdd")
list = rdd1.foreach(fun)
print(list)

rdd2 = sc.parallelize([1, 2, 3, 4, 5])
rdd2.foreach(fun)

#you can only see the print out in executor, not the driver

Print elements of rdd
None


In [39]:
def f(x):
    with open('data/example.txt','a+') as fl:
        print(x,file=fl)
 

open('data/example.txt','w').close()

inputData=sc.parallelize([1,2,3])
inputData.foreach(f)

with open('data/example.txt') as fl:
    print(fl.read())

1
2
3



In [80]:
def f(x):
    with open('data/wc.txt','a+') as fl:
        print(x,file=fl)
 
open('data/wc.txt','w').close()

text=sc.textFile("data/1.txt")
wordCount = text.flatMap(lambda line: line.split(" ")).map(lambda word: (word,1)).reduceByKey(lambda a, b : a + b)

wordCount.foreach(f)
with open('data/wc.txt') as fl:
    print(fl.read())

('0001', 1)
('Lorem', 33)
('Ipsum', 26)
('is', 14)
('simply', 4)
('dummy', 4)
('text', 4)
('of', 42)
('the', 46)
('printing', 2)
('and', 20)
('typesetting', 2)
('industry.', 2)
('has', 8)
('been', 2)
("industry's", 2)
('standard', 4)
('ever', 2)
('since', 4)
('1500s,', 2)
('when', 4)
('an', 2)
('unknown', 2)
('printer', 2)
('took', 2)
('a', 30)
('galley', 2)
('type', 4)
('scrambled', 2)
('it', 8)
('to', 14)
('make', 2)
('specimen', 2)
('book.', 2)
('It', 10)
('survived', 2)
('not', 4)
('only', 2)
('five', 2)
('centuries,', 2)
('but', 4)
('also', 4)
('leap', 2)
('into', 2)
('electronic', 2)
('typesetting,', 2)
('remaining', 2)
('essentially', 2)
('unchanged.', 2)
('was', 2)
('popularised', 2)
('in', 23)
('1960s', 2)
('with', 6)
('release', 2)
('Letraset', 2)
('sheets', 2)
('containing', 2)
('passages,', 2)
('more', 4)
('recently', 2)
('desktop', 4)
('publishing', 4)
('software', 2)
('like', 4)
('Aldus', 2)
('PageMaker', 2)
('including', 2)
('versions', 6)
('Ipsum.', 2)
('0002', 1)
('Con

## 10 saveAsTextFile  and saveAsSequenceFile
Syntax: 
       RDD.saveAsTextFile(path)
       RDD.saveAsSequenceFile(path)


In [81]:
#save as text file in folder: wc_txt
wordCount.saveAsTextFile('data/wc_txt')
#save as binary file in the folder: wc_seq
wordCount.saveAsSequenceFile('data/wc_seq') 

# You can also submit the Spark job to the Cluster Master

The following part is for spark cluster only

In [39]:
#from pyspark.sql import SparkSession

#spark = SparkSession.builder.master('spark://johnzhang10:7077').appName('MyAccumulate').getOrCreate()
#sc = spark.sparkContext

#rdd = sc.parallelize(range(100000))
#rdd.sum()